<a href="https://colab.research.google.com/github/SamanaDahal1/AI-and-ML/blob/main/Week8(Workshop8)TextClassificationQuestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import re
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

In [5]:
data = pd.read_csv("/content/trum_tweet_sentiment_analysis.csv", encoding="ISO-8859-1")

data = data[["text", "Sentiment"]]
data = data.rename(columns={"Sentiment": "label"})

data.head()

,text,label
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [6]:
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()


def text_cleaning_pipeline(dataset, rule="lemmatize"):
    data = dataset.lower()

    # remove URLs
    data = re.sub(r'http\S+|www\S+|https\S+', '', data)

    # remove mentions (@user)
    data = re.sub(r'@\w+', '', data)

    # remove emojis / non-ASCII
    data = re.sub(r'[^\x00-\x7F]+', '', data)

    # remove punctuation & special characters
    data = re.sub(r'[^a-zA-Z\s]', '', data)

    # tokenization
    tokens = data.split()

    # remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # stemming or lemmatization
    if rule == "lemmatize":
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    elif rule == "stem":
        tokens = [stemmer.stem(word) for word in tokens]
    else:
        print("Pick between lemmatize or stem")
        return ""

    return " ".join(tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [8]:
data["clean_text"] = data["text"].astype(str).apply(text_cleaning_pipeline)

In [9]:
X = data["clean_text"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [10]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [11]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [12]:
y_pred = model.predict(X_test_tfidf)

In [13]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.96      0.95    248563
           1       0.91      0.87      0.89    121462

    accuracy                           0.93    370025
   macro avg       0.92      0.91      0.92    370025
weighted avg       0.93      0.93      0.93    370025

